# 10 — GPT-2 124M and llm.c map

**Before:** notebooks **1–9** (tiny GPT track).

**This notebook:** GPT-2 hyperparameters + parameter count vs our tiny model.

Cell 2 allocates ~124M params in RAM (no training) — skip on very low-memory hosts.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import data_path, checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
from llmc.model import GPTConfig

cfg = GPTConfig.gpt2_small(vocab_size=50257, block_size=1024)
print("GPT-2 small config:")
print(f"  layers={cfg.n_layer}, heads={cfg.n_head}, embd={cfg.n_embd}")
print(f"  block_size={cfg.block_size}, vocab={cfg.vocab_size}")


In [ ]:
print("""
Our notebooks (PyTorch, educational)     llm.c (C/CUDA, production speed)
------------------------------------     ---------------------------------
1.Tokens                                 dev/data/*.py tokenizes to .bin
2.Bigram                                 (baseline)
3.Embeddings                             wte + wpe in GPT
4.Attention                              dev/cuda attention kernels
5.GPTBlock                               one transformer block
6.GPT                                    full model forward
7.BatchAndLoss                           cross-entropy + batches
8.Train                                  train_gpt2.cu main loop
9.Sample                                 inference / generation
10.GPT2AndLlmc                           GPT-2 124M + compare checkpoints

Upstream: https://github.com/karpathy/llm.c
Clone llm.c separately for CUDA training; use THIS repo to understand the math.
""")


In [ ]:
from llmc.model import GPT
from llmc.data import CharTokenizer, load_text

text = load_text(DATA)
tok = CharTokenizer.from_text(text)
tiny = GPT(GPTConfig.tiny(tok.vocab_size, 64))
gpt2 = GPT(GPTConfig.gpt2_small(50257, 1024))
print(f"tiny params:   {tiny.count_parameters():,}")
print(f"gpt2 params:   {gpt2.count_parameters():,}")
